In [29]:
import google.generativeai as genai
import json

# DUMMY DATA: This mimics exactly what Stella's test_easy.json will look like.
dummy_doc_id = "12345"
dummy_text = "The patient developed severe hepatotoxicity after taking acetaminophen for their fever."

In [30]:
import google.generativeai as genai
import json
import os
from dotenv import load_dotenv

# 1. Load the hidden variables from the .env file
load_dotenv()

# 2. Grab the key securely
my_api_key = os.environ.get("GEMINI_API_KEY")

if not my_api_key:
    raise ValueError("API key not found! Check your .env file.")

# 3. Configure the AI using the secure key
genai.configure(api_key=my_api_key)

In [31]:
model = genai.GenerativeModel('gemini-2.5-flash')

# The exact schema we want Gemini to follow
generation_config = genai.types.GenerationConfig(
    temperature=0.0, # Zero creativity, strictly factual extraction
    response_mime_type="application/json", # Forces the API to return valid JSON
)

In [32]:
system_instruction = """
You are a highly precise medical extraction AI. 
Find all diseases and chemicals in the provided text. 
Return ONLY a valid JSON object with two arrays: "diseases" and "chemicals".
If none are found, return empty arrays.
"""

# Combine the instruction with Stella's (dummy) text
full_prompt = f"{system_instruction}\n\nText to analyze: {dummy_text}"

In [33]:
print("Sending request to Gemini API...")
response = model.generate_content(
    full_prompt,
    generation_config=generation_config
)

# Parse the text response into a Python dictionary
extracted_data = json.loads(response.text)

# Format the final output to match your contract
final_output = {
    "doc_id": dummy_doc_id,
    "gemini_response": extracted_data
}

# Save it to a file
with open("gemini_raw_output.json", "w") as outfile:
    json.dump(final_output, outfile, indent=2)

print("Success! Data saved to gemini_raw_output.json")
print(json.dumps(final_output, indent=2))

Sending request to Gemini API...
Success! Data saved to gemini_raw_output.json
{
  "doc_id": "12345",
  "gemini_response": {
    "diseases": [
      "hepatotoxicity"
    ],
    "chemicals": [
      "acetaminophen"
    ]
  }
}
